### Agentic Synthetic Data Generation

In [12]:
import pandas as pd
import numpy as np
import random
import os
import torch
import faiss
import json
import re
from tqdm import tqdm
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM
logging.set_verbosity_error()
from beir.datasets.data_loader import GenericDataLoader
from dotenv import load_dotenv
from huggingface_hub import login

In [4]:
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split='train')

100%|██████████| 8841823/8841823 [01:18<00:00, 112226.23it/s]


In [6]:
load_dotenv('/work/mbouthil/MMATH-CM-Research-Project/token.env')
token = os.getenv('HUGGINGFACE_TOKEN')
login(token=token)

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token
)

Loading weights: 100%|██████████| 291/291 [00:01<00:00, 146.08it/s, Materializing param=model.norm.weight]                              


In [22]:
q_ids = [key for key in queries.keys()]
q_id = random.sample(q_ids, 1)[0]

r_query = queries[q_id]
print(r_query)

how does cobra insurance work between jobs


In [23]:
system_prompt = '''
You are a helpful AI Assistant. You are to follow the following instructions:

You will be given a query. Your task is to think about how you would produce a new query asking the same underlying question.

Provide your thoughts:
'''

In [24]:
def llm_call(notes: list[str], system_prompt:str) -> list[str]:
    messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note}
        ]
        for note in notes
    ]

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_p=0.9,
            do_sample=True
        )

    responses = []
    for i in range(len(notes)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

In [25]:
response = llm_call([r_query], system_prompt)

In [26]:
print(response[0])

assistant

To ask the same underlying question in a different way, I would rephrase the query as follows:

"What are the specifics of Cobra insurance coverage and how does it apply to individuals who are transitioning between jobs or employment statuses?"

This new query still conveys the same question about Cobra insurance, but frames it in a more specific and detailed manner, focusing on the aspects of job transitions and employment status changes.


In [34]:
example = response[0][10:]
print(example)


To ask the same underlying question in a different way, I would rephrase the query as follows:

"What are the specifics of Cobra insurance coverage and how does it apply to individuals who are transitioning between jobs or employment statuses?"

This new query still conveys the same question about Cobra insurance, but frames it in a more specific and detailed manner, focusing on the aspects of job transitions and employment status changes.


In [37]:
system_prompt = f'''
You are a helpful AI Assistant. Your task is to create a new question from the provided query, asking the same underlying infromation. 

As an example:
{example}
'''

In [38]:
print(system_prompt)


You are a helpful AI Assistant. Your task is to create a new question from the provided query, asking the same underlying infromation. 

As an example:

To ask the same underlying question in a different way, I would rephrase the query as follows:

"What are the specifics of Cobra insurance coverage and how does it apply to individuals who are transitioning between jobs or employment statuses?"

This new query still conveys the same question about Cobra insurance, but frames it in a more specific and detailed manner, focusing on the aspects of job transitions and employment status changes.



In [39]:
response = llm_call([r_query], system_prompt)

In [43]:
print(response[0][10:])


"What are the details of the process for enrolling in and maintaining COBRA insurance coverage when transitioning from one job to another, including any time limits and requirements for eligibility?"


### Creates new Complex query asking the same question! 

# Judge LLM

In [44]:
system_prompt = f'''
You are a helpful AI Assistant.

Your task is to judge whether both provided queries pose the same underlying question. 

If they do not, please answer with FALSE. 
'''

In [56]:
judge_query = [str('"' + queries[q_id] + '"\n\nand \n' + response[0][10:])]

In [57]:
print(judge_query[0])

"how does cobra insurance work between jobs"

and 

"What are the details of the process for enrolling in and maintaining COBRA insurance coverage when transitioning from one job to another, including any time limits and requirements for eligibility?"


In [58]:
judge_answer = llm_call(judge_query, system_prompt=system_prompt)

In [59]:
print(judge_answer[0])

assistant

TRUE. 

Both queries are asking about the process of enrolling in and maintaining COBRA (Consolidated Omnibus Budget Reconciliation Act) insurance coverage when transitioning from one job to another.


In [62]:
for answer in judge_answer:
    if 'TRUE' in answer:
        print(answer)

assistant

TRUE. 

Both queries are asking about the process of enrolling in and maintaining COBRA (Consolidated Omnibus Budget Reconciliation Act) insurance coverage when transitioning from one job to another.


# Putting it all together

In [63]:
# Loading Packages
import pandas as pd
import numpy as np
import random
import os
import torch
import faiss
import json
import re
from tqdm import tqdm
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM
logging.set_verbosity_error()
from beir.datasets.data_loader import GenericDataLoader
from dotenv import load_dotenv
from huggingface_hub import login

In [64]:
# Loading Data
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split='train')

100%|██████████| 8841823/8841823 [01:15<00:00, 117033.08it/s]


In [66]:
# Additional Synthetic Queries to be created
N=100_000

# Copying Dataset
t_corpus = corpus.copy()
t_queries = queries.copy()
t_qrels = qrels.copy()

# Gathering Ids and subset
max_id = int(max([int(key) for key in t_queries.keys()]))
q_subset = [(keys, values) for keys, values in queries.items()][:N]

In [65]:
# Loading LLM
load_dotenv('/work/mbouthil/MMATH-CM-Research-Project/token.env')
token = os.getenv('HUGGINGFACE_TOKEN')
login(token=token)

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token
)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 294.58it/s, Materializing param=model.norm.weight]                              


In [123]:
cot_prompt='''
You are a helpful AI assistant. You are to follow the following instructions:

You will be given a query. Your task is to think about how you would produce a new query asking the same underlying question.

Proivide your thoughts:
'''

In [124]:
creation_prompt=f'''
You are a helpful AI assistant. Your task is to create a new question from the provided query, asking the same underlying infromation. 

Provide only your new question.  

Moreover, consider the following:
'''

In [125]:
print(creation_prompt + example)


You are a helpful AI assistant. Your task is to create a new question from the provided query, asking the same underlying infromation. 

Provide only your new question.  

Moreover, consider the following:

To ask the same underlying question in a different way, I would rephrase the query as follows:

"What are the specifics of Cobra insurance coverage and how does it apply to individuals who are transitioning between jobs or employment statuses?"

This new query still conveys the same question about Cobra insurance, but frames it in a more specific and detailed manner, focusing on the aspects of job transitions and employment status changes.


In [126]:
judge_prompt='''
You are a helpful AI assistant. 

Your task is to judge whether both provided queries pose the same underlying question. 

Ensure that your answer contains TRUE or FALSE. 
'''

In [127]:
### LLM function ###
def llm_pass(
        messages:list[list[dict]],
        padding:bool=True,
        truncation:bool=True,
        max_tokens:int=256, 
        temp:float=0.1,
        top_p:float=0.9,
) -> list[str]:

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=padding,
        truncation=truncation
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temp,
            top_p=top_p,
            do_sample=True
        )

    responses = []
    for i in range(len(messages)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

In [128]:
### Creating Batches ###
def batch_splits(queries:list, batch_size:int=64) -> list[tuple]:

    for i in range(0, len(queries), batch_size):
        yield queries[i:i + batch_size]

batches = batch_splits(q_subset)

In [129]:
for batch in batches:
    # print(type(batch[0]))
    ids, queries = batch[0]
    ids = [ids]
    queries = [queries]

    print(ids)

    break

['1185869']


In [ ]:
### Creating new queries ###
new_queries = []
old_ids = []

for batch in batches:

    # ids, queries = batch[:3]
    # # ids = [ids]
    # # queries = [queries]
    # N = len(ids)

    ids, queries = zip(*batch)
    N = len(ids)

    # Step 1
    cot_messages = [
        [
            {"role": "system", "content": cot_prompt},
            {"role": "user", "content": query}
        ]
        for query in queries
    ]
    cot_responses = llm_pass(cot_messages)
    cot_responses = [response[11:] for response in cot_responses]

    # Step 2
    creation_messages = [
        [
            {"role": "system", "content": creation_prompt + cot_responses[i]},
            {"role": "user", "content": query}
        ]
        for i, query in enumerate(queries)
    ]
    new_queries = llm_pass(creation_messages)
    new_queries = [query[11:] for query in new_queries]

    # Step 3
    judge_messages = [
        [
            {"role": "system", "content": judge_prompt},
            {"role": "user", "content": 
             str('"' + queries[i] + '"\n\nand \n' + new_queries[i])
             }
        ]
        for i in range(N)
    ]
    verdicts = llm_pass(judge_messages)
    bool_verdicts = [1 if 'TRUE' in answer else 0 for answer in verdicts]

    break

In [138]:
print(queries[0])
print("\n")
print(cot_responses[0])
print("\n")
print(new_queries[0])
print("\n")
print(bool_verdicts)

ibm filenet discovery


To produce a new query asking the same underlying question, I would rephrase the query as follows:

"What is IBM FileNet Discovery, and what are its key features and applications?"

This new query still asks about IBM FileNet Discovery, but it also includes additional information that I would like to know, such as its key features and applications. This will help me to better understand the topic and potentially get more comprehensive information.

Alternatively, I could also ask:

* "What is the purpose of IBM FileNet Discovery in the context of enterprise content management?"
* "How does IBM FileNet Discovery differ from other content management systems?"
* "What are the benefits of using IBM FileNet Discovery for document and data management?"
* "Can you provide an overview of IBM FileNet Discovery's architecture and functionality?"

These new queries still ask about IBM FileNet Discovery, but from different angles and with different focuses.


What are the p

In [139]:
print(new_queries[0])

What are the primary functions and capabilities of IBM FileNet Discovery in the context of enterprise content management systems?
